In [ ]:
!pip install -q pandas requests openpyxl tqdm python-levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 71.2 MB/s eta 0:00:00


In [ ]:
import getpass
TMDB_KEY = getpass.getpass("TMDb API key (v3): ")
OMDB_KEY = getpass.getpass("OMDb API key (enter para saltar): ").strip() or None

TMDb API key (v3): ··········
OMDb API key (enter para saltar): ··········


In [ ]:
from google.colab import files
uploaded = files.upload()
FNAME = list(uploaded.keys())[0]   # e.g. "enriched_DATA SET SEM DAYS OF THE WEEK.xlsx"
FNAME

Saving enriched_DATA SET SEM DAYS OF THE WEEK.xlsx to enriched_DATA SET SEM DAYS OF THE WEEK.xlsx


'enriched_DATA SET SEM DAYS OF THE WEEK.xlsx'

In [ ]:
import pandas as pd, requests, time, re, os
from tqdm import tqdm
from Levenshtein import ratio as lev_ratio

# ---------- Config ----------
SHEET_NAME = None            # None => 1ª sheet
SLEEP = 0.18                 # backoff simpático
SIM_THRESHOLD = 0.60
USE_DATE_HINT = True
USE_STUDIO_HINT = True

MOVIE_CACHE_PATH   = "movie_cache.csv"     # Title,Year -> tmdb_id, imdb_id, runtime, num_langs, prod_companies JSON
PERSON_CACHE_PATH  = "person_cache.csv"    # person_id  -> birth_year
COMPANY_CACHE_PATH = "company_cache.csv"   # company_id -> origin_country

# ---------- HTTP helper ----------
def http_get(url, params=None, retries=3, timeout=20):
    back = 0.6
    for _ in range(retries):
        try:
            r = requests.get(url, params=params, timeout=timeout)
            if r.status_code in (429,500,502,503,504):
                time.sleep(back); back *= 2; continue
            return r
        except requests.RequestException:
            time.sleep(back); back *= 2
    return None

# ---------- Utils ----------
_clean_re = re.compile(r"[^a-z0-9 ]+")
def norm_title(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"\s*\(\d{4}\)$","",s)
    s = _clean_re.sub(" ", s)
    s = re.sub(r"\s+"," ", s)
    return s

def norm_text(s):
    return re.sub(r"\s+"," ", (s or "").strip().lower())

def days_diff(date_hint, tmdb_release_date):
    if date_hint is None or tmdb_release_date is None: return None
    try:
        d1 = pd.to_datetime(date_hint, errors="coerce")
        d2 = pd.to_datetime(tmdb_release_date, errors="coerce")
        if pd.isna(d1) or pd.isna(d2): return None
        return abs((d2 - d1).days)
    except: return None

def studio_sim(studio_hint, tmdb_companies):
    if not studio_hint or not tmdb_companies: return 0.0
    sh = norm_text(studio_hint)
    best = 0.0
    for c in tmdb_companies:
        nm = norm_text(c.get("name"))
        if nm:
            best = max(best, lev_ratio(sh, nm))
    return best

def get_row_year(row):
    y = None
    if "Year" in row and pd.notna(row["Year"]):
        try: y = int(row["Year"])
        except: y = None
    if y is None and "Date" in row and pd.notna(row["Date"]):
        try:
            y = pd.to_datetime(row["Date"], errors="coerce").year
            if pd.isna(y): y = None
        except: y = None
    return int(y) if y is not None else None

# ---------- Caches persistentes ----------
def load_csv_cache(path, key_cols):
    if os.path.exists(path):
        try:
            df = pd.read_csv(path, dtype="string")
            for c in key_cols:
                if c not in df.columns: df[c] = pd.NA
            return df
        except: pass
    return pd.DataFrame(columns=key_cols, dtype="string")

movie_cache  = load_csv_cache(MOVIE_CACHE_PATH, ["title_norm","year","tmdb_id","imdb_id","runtime","num_langs","prod_companies_json"])
person_cache = load_csv_cache(PERSON_CACHE_PATH, ["person_id","birth_year"])
company_cache= load_csv_cache(COMPANY_CACHE_PATH,["company_id","origin_country"])

def save_caches():
    movie_cache.to_csv(MOVIE_CACHE_PATH, index=False)
    person_cache.to_csv(PERSON_CACHE_PATH, index=False)
    company_cache.to_csv(COMPANY_CACHE_PATH, index=False)

def cache_get_movie(title_norm, year):
    hit = movie_cache[(movie_cache["title_norm"]==title_norm) & (movie_cache["year"]==(str(year) if year is not None else pd.NA))]
    return hit.iloc[0] if not hit.empty else None

def cache_put_movie(title_norm, year, tmdb_id, imdb_id, runtime, num_langs, prod_companies_json):
    global movie_cache
    movie_cache = movie_cache[
        ~((movie_cache["title_norm"]==title_norm) & (movie_cache["year"]==(str(year) if year is not None else pd.NA)))
    ]
    movie_cache.loc[len(movie_cache)] = {
        "title_norm": title_norm,
        "year": (str(year) if year is not None else pd.NA),
        "tmdb_id": str(tmdb_id) if tmdb_id else pd.NA,
        "imdb_id": imdb_id or pd.NA,
        "runtime": str(runtime) if runtime is not None else pd.NA,
        "num_langs": str(num_langs) if num_langs is not None else pd.NA,
        "prod_companies_json": prod_companies_json or "[]"
    }

def cache_get_person_birth(person_id):
    if not person_id: return None
    hit = person_cache[person_cache["person_id"]==str(person_id)]
    if hit.empty: return None
    by = hit.iloc[0]["birth_year"]
    return int(by) if pd.notna(by) and str(by).isdigit() else None

def cache_put_person_birth(person_id, birth_year):
    global person_cache
    if not person_id: return
    person_cache = person_cache[person_cache["person_id"]!=str(person_id)]
    person_cache.loc[len(person_cache)] = {
        "person_id": str(person_id),
        "birth_year": str(birth_year) if birth_year is not None else pd.NA
    }

def cache_get_company_country(company_id):
    if not company_id: return None
    hit = company_cache[company_cache["company_id"]==str(company_id)]
    if hit.empty: return None
    oc = hit.iloc[0]["origin_country"]
    return oc if pd.notna(oc) else None

def cache_put_company_country(company_id, origin_country):
    global company_cache
    if not company_id: return
    company_cache = company_cache[company_cache["company_id"]!=str(company_id)]
    company_cache.loc[len(company_cache)] = {
        "company_id": str(company_id),
        "origin_country": origin_country or pd.NA
    }

# ---------- TMDb search/pick ----------
def tmdb_search_candidates(title, year):
    norm = norm_title(title)
    def q(query, y):
        p = {"api_key": TMDB_KEY, "query": query, "include_adult":"false"}
        if y is not None: p["year"] = int(y)
        r = http_get("https://api.themoviedb.org/3/search/movie", p)
        if not r or r.status_code>=400: return []
        return r.json().get("results", []) or []
    years = [year] if year is not None else [None]
    if year is not None: years += [year-1, year+1]
    queries = {title, norm}
    cands, seen = [], set()
    for y in years:
        for qstr in queries:
            for it in q(qstr, y):
                k = it.get("id")
                if k not in seen:
                    seen.add(k); cands.append(it)
    return cands

def composite_score(item, want_title, want_year, date_hint=None):
    vc = item.get("vote_count",0) or 0
    pop = item.get("popularity",0.0) or 0.0
    rd  = item.get("release_date") or ""
    yr  = int(rd[:4]) if len(rd)>=4 and rd[:4].isdigit() else None
    y_pen = abs(yr - want_year) if (want_year and yr) else 0
    sim = lev_ratio(norm_title(item.get("title","")), norm_title(want_title))
    is_doc = 99 in (item.get("genre_ids") or [])
    score = (vc*12) + (pop*2) + (sim*8) - (y_pen*2) - (20 if is_doc else 0)
    if date_hint is not None:
        dd = days_diff(date_hint, rd)
        if dd is not None:
            if dd <= 7: score += 15
            elif dd <= 30: score += 8
            elif dd <= 60: score += 4
    return score, sim

def pick_best_tmdb(title, year, date_hint=None, studio_hint=None):
    cands = tmdb_search_candidates(title, year)
    if not cands: return None
    non_docs = [x for x in cands if 99 not in (x.get("genre_ids") or [])]
    pool = non_docs if non_docs else cands
    pool.sort(key=lambda x: composite_score(x, title, year, date_hint if USE_DATE_HINT else None), reverse=True)
    top = pool[0]
    _, sim = composite_score(top, title, year, date_hint if USE_DATE_HINT else None)
    if sim < SIM_THRESHOLD: return None
    # Se temos pista de Studio, validar top-3 com empresas
    if USE_STUDIO_HINT and studio_hint:
        best = None; best_aug = -1e9
        for cand in pool[:3]:
            mid = cand.get("id")
            r = http_get(f"https://api.themoviedb.org/3/movie/{mid}", {"api_key": TMDB_KEY})
            companies = (r.json().get("production_companies") or []) if (r and r.status_code<400) else []
            base,_ = composite_score(cand, title, year, date_hint)
            ss = studio_sim(studio_hint, companies)
            aug = base + ss*20
            if aug > best_aug:
                best_aug = aug; best = cand
        return best
    return top

# ---------- TMDb details ----------
def tmdb_movie_core(tmdb_id):
    out = {"runtime":None,"spoken_languages":[],"production_companies":[],"imdb_id":None,
           "actor_id":None,"crew":[]}
    # credits
    r = http_get(f"https://api.themoviedb.org/3/movie/{tmdb_id}/credits", {"api_key": TMDB_KEY})
    if r and r.status_code<400:
        j = r.json()
        out["crew"] = j.get("crew") or []
        cast = j.get("cast") or []
        if cast:
            main = sorted(cast, key=lambda x: x.get("order",9999))[0]
            out["actor_id"] = main.get("id")
    # details
    r = http_get(f"https://api.themoviedb.org/3/movie/{tmdb_id}", {"api_key": TMDB_KEY})
    if r and r.status_code<400:
        j = r.json()
        out["runtime"] = j.get("runtime")
        out["spoken_languages"] = j.get("spoken_languages") or []
        out["production_companies"] = j.get("production_companies") or []
    # ext ids
    r = http_get(f"https://api.themoviedb.org/3/movie/{tmdb_id}/external_ids", {"api_key": TMDB_KEY})
    if r and r.status_code<400:
        out["imdb_id"] = r.json().get("imdb_id")
    return out

def tmdb_person_birth_year(person_id):
    # cache
    by = cache_get_person_birth(person_id)
    if by is not None: return by
    r = http_get(f"https://api.themoviedb.org/3/person/{person_id}", {"api_key": TMDB_KEY})
    if not r or r.status_code>=400:
        cache_put_person_birth(person_id, None); return None
    dob = (r.json().get("birthday") or "")
    year = int(dob[:4]) if len(dob)>=4 and dob[:4].isdigit() else None
    cache_put_person_birth(person_id, year)
    time.sleep(SLEEP)
    return year

def tmdb_company_country(company_id):
    # usa cache; se faltar, chama /company/{id}
    oc = cache_get_company_country(company_id)
    if oc is not None: return oc
    r = http_get(f"https://api.themoviedb.org/3/company/{company_id}", {"api_key": TMDB_KEY})
    if not r or r.status_code>=400:
        cache_put_company_country(company_id, None); return None
    oc = r.json().get("origin_country")
    cache_put_company_country(company_id, oc)
    time.sleep(SLEEP)
    return oc

def omdb_runtime_languages(imdb_id):
    if not OMDB_KEY or not imdb_id: return None, None
    r = http_get("https://www.omdbapi.com/", {"i":imdb_id,"apikey":OMDB_KEY})
    if not r or r.status_code>=400 or r.json().get("Response")!="True":
        return None, None
    rt = r.json().get("Runtime")
    runtime = None
    if rt and "min" in rt:
        try: runtime = int(rt.split(" ")[0])
        except: pass
    lang = r.json().get("Language")
    num_langs = len([x.strip() for x in lang.split(",") if x.strip()]) if lang else None
    return runtime, num_langs

def pick_main_producer_id(crew):
    # prioridade Producer -> Co-Producer -> Executive Producer
    order = ["producer","co-producer","executive producer"]
    best = None
    for role in order:
        for p in crew or []:
            if (p.get("job") or "").strip().lower() == role:
                return p.get("id")
    # fallback nulo
    return best

# ---------- Process ----------
# ler df SEM mexer em colunas existentes
xls = pd.ExcelFile(FNAME)
sheet = SHEET_NAME or xls.sheet_names[0]
df = pd.read_excel(FNAME, sheet_name=sheet)

# garantir que as 4 colunas existem (se não, criar vazias)
target_cols = ["Runtime (min)","Production Company Country","Lead Actor Birth Year","Lead Producer Birth Year"]
for c in target_cols:
    if c not in df.columns: df[c] = None
df[target_cols] = df[target_cols].astype("object")

# escolher coluna Studio se existir (ajuda a escolher candidato)
studio_col = None
for cand in ["Studio","Distributor","Studio/Dist","Estudio","Estúdio"]:
    if cand in df.columns:
        studio_col = cand; break

# loop só para LINHAS com valores em falta nas 4 colunas
for i in tqdm(range(len(df))):
    need_runtime   = pd.isna(df.at[i,"Runtime (min)"])
    need_country   = pd.isna(df.at[i,"Production Company Country"])
    need_actor_by  = pd.isna(df.at[i,"Lead Actor Birth Year"])
    need_produc_by = pd.isna(df.at[i,"Lead Producer Birth Year"])
    if not (need_runtime or need_country or need_actor_by or need_produc_by):
        continue  # já completo

    title = str(df.at[i,"Title"]).strip() if "Title" in df.columns else None
    if not title:
        continue
    year  = get_row_year(df.loc[i])
    date_hint = df.at[i,"Date"] if (USE_DATE_HINT and "Date" in df.columns) else None
    studio_hint = str(df.at[i,studio_col]) if (USE_STUDIO_HINT and studio_col and pd.notna(df.at[i,studio_col])) else None

    key = (norm_title(title), year)
    # tenta cache de movie
    hit = cache_get_movie(key[0], key[1])
    if hit is not None:
        tmdb_id = int(hit["tmdb_id"]) if pd.notna(hit["tmdb_id"]) else None
        imdb_id = hit["imdb_id"] if pd.notna(hit["imdb_id"]) else None
        runtime = int(hit["runtime"]) if pd.notna(hit["runtime"]) and str(hit["runtime"]).isdigit() else None
        num_langs = int(hit["num_langs"]) if pd.notna(hit["num_langs"]) and str(hit["num_langs"]).isdigit() else None
        import json
        try:
            prod_companies = json.loads(hit["prod_companies_json"] or "[]")
        except:
            prod_companies = []
    else:
        # pick + details fresh
        pick = pick_best_tmdb(title, year, date_hint, studio_hint)
        if not pick:
            # não deu → não atualiza nada nesta linha
            continue
        tmdb_id = pick.get("id")
        det = tmdb_movie_core(tmdb_id)
        imdb_id = det.get("imdb_id")
        runtime = det.get("runtime")
        num_langs = len(det.get("spoken_languages") or [])
        prod_companies = det.get("production_companies") or []
        # guarda no cache persistente
        import json
        cache_put_movie(key[0], key[1], tmdb_id, imdb_id, runtime, num_langs, json.dumps(prod_companies))
        save_caches()
        time.sleep(SLEEP)

    # ---- Runtime (TMDb → OMDb fallback) ----
    if need_runtime:
        rt = runtime
        if rt is None and imdb_id:
            rt2, _ = omdb_runtime_languages(imdb_id)
            if rt2 is not None: rt = rt2
        if rt is not None:
            df.at[i,"Runtime (min)"] = int(rt)

    # ---- Production Company Country ----
    if need_country:
        country = None
        # 1) tentar pelo 1º estúdio listado no filme
        if prod_companies:
            pc0 = prod_companies[0]
            country = pc0.get("origin_country")
            if (not country) and pc0.get("id"):
                # 2) chamar /company/{id} (com cache)
                country = tmdb_company_country(pc0.get("id"))
        if country:
            df.at[i,"Production Company Country"] = country

    # ---- Lead Actor Birth Year ----
    if need_actor_by:
        # se movie_cache não guardou actor_id, vamos buscar (uma vez)
        if 'det' not in locals():
            det = tmdb_movie_core(tmdb_id) if tmdb_id else {}
        actor_id = det.get("actor_id")
        by = tmdb_person_birth_year(actor_id) if actor_id else None
        if by is not None:
            df.at[i,"Lead Actor Birth Year"] = int(by)

    # ---- Lead Producer Birth Year ----
    if need_produc_by:
        if 'det' not in locals():
            det = tmdb_movie_core(tmdb_id) if tmdb_id else {}
        crew = det.get("crew") or []
        prod_id = None
        # prioridade Producer -> Co-Producer -> Executive Producer
        for role in ["producer","co-producer","executive producer"]:
            for p in crew:
                if (p.get("job") or "").strip().lower() == role:
                    prod_id = p.get("id"); break
            if prod_id: break
        byp = tmdb_person_birth_year(prod_id) if prod_id else None
        if byp is not None:
            df.at[i,"Lead Producer Birth Year"] = int(byp)

# exporta SEM mexer nas colunas originais
OUTFILE = f"enriched_{FNAME}"
df.to_excel(OUTFILE, index=False)
save_caches()
print("✅ Done →", OUTFILE)


100%|██████████| 34567/34567 [1:18:05<00:00,  7.38it/s]


✅ Done → enriched_enriched_DATA SET SEM DAYS OF THE WEEK.xlsx


In [ ]:
from google.colab import files
files.download(OUTFILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>